# Obstore backend

`zarr-vectors` can write and read ZV stores through any
`zarr.abc.store.Store` implementation.  The cleanest path for cloud
object storage is `backend="obstore"`, which dispatches on URL scheme
(`s3://`, `gs://`, `az://`, `http(s)://`, `file://`) to the matching
`obstore.store.*` backend and wraps it in `zarr.storage.ObjectStore`.

Three patterns are shown below:

1. **URL + `backend="obstore"`** — the common case.
2. **Pre-built `obstore.store.*` object** — pass any obstore store
   directly (custom credentials, custom retry config, etc.).
3. **Pre-built `zarr.abc.store.Store`** — full StoreLike pass-through
   (works for `MemoryStore`, `FsspecStore`, anything).

In [2]:
import numpy as np
from zarr_vectors import open_store
from zarr_vectors.types.points import read_points, write_points
from zarr_vectors.core.metadata import NgffAxis


## 1. URL + `backend="obstore"`

Replace `STORE` with your own bucket.  Credentials are picked up from
the standard SDK environment (e.g. `GOOGLE_APPLICATION_CREDENTIALS` for
GCS, `AWS_ACCESS_KEY_ID` for S3); they can also be passed explicitly as
`backend_kwargs` (forwarded to the underlying `obstore.store.*`
constructor).

In [5]:
rng = np.random.default_rng(42)

N = 1_200_000
# Positions: 200k points in a 1000³ µm synchrotron scan volume
positions   = rng.uniform(0, 1000, (N, 3)).astype(np.float32)

# Per-vertex attributes
intensity   = rng.uniform(0, 1, N).astype(np.float32)          # absorption value
label       = rng.integers(0, 8, N).astype(np.int32)            # tissue class
confidence  = rng.uniform(0.5, 1, N).astype(np.float32)

print(f"positions : {positions.shape}  dtype={positions.dtype}")
print(f"intensity : {intensity.shape}  range [{intensity.min():.3f}, {intensity.max():.3f}]")
print(f"label     : {label.shape}  classes {np.unique(label)}")


positions : (1200000, 3)  dtype=float32
intensity : (1200000,)  range [0.000, 1.000]
label     : (1200000,)  classes [0 1 2 3 4 5 6 7]


In [6]:
STORE = "gs://allen_neuroglancer_ccf/zarrvectors/obstore5"

from zarr_vectors.types.points import write_points
from zarr_vectors import create_store


store = create_store(STORE,
                     bounds = ([0, 0, 0], [1000, 1000, 1000]),
                     chunk_shape=(200.0, 200.0, 200.0),   # 200³ µm per chunk → 125 chunks
                     axes=[NgffAxis(name="x", type="space"),
                           NgffAxis(name="y", type="space"),
                           NgffAxis(name="z", type="space")],)
write_points(
    store,
    positions,
    attributes={
        "intensity":  intensity,
        "label":      label,
        "confidence": confidence,
    },
)
print("Write complete.")


Write complete.


In [7]:
# mode='r' yields a strictly read-only zarr store (writes raise).
store = open_store(STORE, mode="r")
print("read_only:", store._zarr.store.read_only)

result = read_points(STORE)
print("vertex_count:", result["vertex_count"])
print("positions shape:", result["positions"].shape)

read_only: True
vertex_count: 1200000
positions shape: (1200000, 3)


In [8]:
result

{'positions': array([[773.95605 , 438.87845 , 858.5979  ],
        [697.36804 ,  94.177345, 975.6224  ],
        [761.1397  , 786.06433 , 128.11363 ],
        ...,
        [844.7998  ,  70.01878 , 508.20618 ],
        [135.45544 , 375.29605 , 958.645   ],
        [643.6087  , 547.94385 , 827.1307  ]],
       shape=(1200000, 3), dtype=float32),
 'attributes': {},
 'vertex_count': 1200000}

## 2. Pre-built `obstore.store.*` object

When you need fine-grained control over the obstore client (custom
retry policy, credential provider, anonymous access, etc.), construct
the `obstore.store.*` instance yourself and pass it as `path`.

In [ ]:
from obstore.store import GCSStore

gcs = GCSStore.from_url(STORE)
store = open_store(gcs, mode="r")
print("read_only:", store._zarr.store.read_only)
print("root attrs keys:", list(store.attrs.to_dict().keys()))

## 3. Pre-built `zarr.abc.store.Store`

Any object that satisfies the `zarr.abc.store.Store` interface can be
passed directly.  This includes `zarr.storage.MemoryStore`,
`zarr.storage.FsspecStore`, and `zarr.storage.LocalStore`.

In [ ]:
from zarr.storage import MemoryStore

mem = MemoryStore()
write_points(mem, positions=positions[:50])
result = read_points(mem)
print("in-memory vertex_count:", result["vertex_count"])